# ETL Silver - ECMWF CF (recorte al poligono de la cuenca)

Toma Bronze (todo el bounding box, ya limitado server-side) y se queda solo con los puntos de grilla dentro del buffer (~0.15 grados) del poligono union de las 3 sub-cuencas, tageando cada punto con `subcuenca_id`/`subcuenca_nombre`.

In [ ]:
# restartPython() no se usa aca: mata la sesion de spark en este compute
# ("spark should be initialized with the first notebook command"). geopandas/pyogrio
# son paquetes puros de Python, quedan importables sin reiniciar el kernel.
%pip install --quiet geopandas pyogrio

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from delta.tables import DeltaTable
from datetime import timedelta, date
from pyspark.sql import functions as F

GEOJSON_PATH = "/Workspace/Users/joaquintschopp@gmail.com/rio-uruguay-hydro-pipeline/SIG/subcuencas_modelo.geojson"
BUFFER_DEG = 0.15  # ~15km, medio ancho de celda 0.25 grados
METRIC_CRS = 32721  # UTM 21S, para medir el buffer en metros de forma mas precisa

sub = gpd.read_file(GEOJSON_PATH)[["fid", "nombre", "geometry"]].rename(columns={"fid": "subcuenca_id", "nombre": "subcuenca_nombre"})
sub_buffered = sub.copy()
sub_buffered["geometry"] = sub.to_crs(METRIC_CRS).buffer(BUFFER_DEG * 111000).to_crs(4326)


def tag_points(pdf: pd.DataFrame) -> pd.DataFrame:
    """Recibe un pandas DataFrame con columnas latitude/longitude (unicas), devuelve
    subcuenca_id/subcuenca_nombre por punto (o None si cae fuera de las 3 con buffer)."""
    uniq = pdf[["latitude", "longitude"]].drop_duplicates()
    gdf_pts = gpd.GeoDataFrame(uniq, geometry=[Point(lo, la) for la, lo in zip(uniq["latitude"], uniq["longitude"])], crs=4326)
    joined = gpd.sjoin(gdf_pts, sub_buffered, how="left", predicate="within").drop_duplicates(subset=["latitude", "longitude"], keep="first")
    tags = joined[["latitude", "longitude", "subcuenca_id", "subcuenca_nombre"]]
    return pdf.merge(tags, on=["latitude", "longitude"], how="left")


In [ ]:
BRONZE_TABLE = 'weather.bronze.ecmwf_forecast_cf'
TARGET_TABLE = 'weather.silver.ecmwf_forecast_cf_basin'

try:
    dbutils.widgets.dropdown('load_mode', 'incremental', ['full', 'incremental', 'backfill'])
    dbutils.widgets.text('incremental_lookback_days', '3')
    dbutils.widgets.text('range_start', '')
    dbutils.widgets.text('range_end', '')
    load_mode = dbutils.widgets.get('load_mode')
    incremental_lookback_days = int(dbutils.widgets.get('incremental_lookback_days'))
    range_start = dbutils.widgets.get('range_start') or None
    range_end = dbutils.widgets.get('range_end') or None
except Exception:
    load_mode = 'incremental'
    incremental_lookback_days = 3
    range_start = None
    range_end = None

print(f'load_mode={load_mode}, incremental_lookback_days={incremental_lookback_days}, range_start={range_start}, range_end={range_end}')

In [ ]:
bronze = spark.table(BRONZE_TABLE)

if load_mode == 'incremental':
    max_target = spark.table(TARGET_TABLE).agg(F.max('run_date').alias('m')).first()['m']
    if max_target is not None:
        start_date = max_target - timedelta(days=incremental_lookback_days)
        bronze = bronze.filter(F.col('run_date') >= F.lit(start_date))
        print(f'Procesando desde {start_date}')
elif load_mode == 'backfill':
    # Modo pensado para la reconstruccion historica (ver Historic_ECMWF_CF): filtra por un
    # rango explicito en vez de por el maximo actual de la tabla Silver, porque el backfill
    # trae filas mas viejas que ese maximo (el modo incremental nunca las agarraria). Se corre
    # una vez por cada lote que aterriza en Bronze, para no hacer un unico toPandas() gigante
    # de todo el historico.
    if not range_start or not range_end:
        dbutils.notebook.exit('load_mode=backfill requiere range_start y range_end (YYYY-MM-DD)')
    bronze = bronze.filter((F.col('run_date') >= F.lit(range_start)) & (F.col('run_date') <= F.lit(range_end)))
    print(f'Procesando rango explicito {range_start} .. {range_end}')

if bronze.limit(1).count() == 0:
    dbutils.notebook.exit('No hay filas nuevas de Bronze para procesar')

pdf = bronze.toPandas()
pdf_tagged = tag_points(pdf)
pdf_tagged = pdf_tagged[pdf_tagged['subcuenca_id'].notna()].copy()
pdf_tagged['subcuenca_id'] = pdf_tagged['subcuenca_id'].astype(int)
print(f'Filas Bronze: {len(pdf)} -> Filas dentro de la cuenca: {len(pdf_tagged)}')

silver_df = (
    spark.createDataFrame(pdf_tagged)
    .withColumn('processed_at', F.current_timestamp())
    .withColumn('updated_at', F.current_timestamp())
    .withColumn('source_table', F.lit(BRONZE_TABLE))
    .select(
        'run_date', 'run_time', 'step_hours', 'valid_date', 'valid_datetime', 'latitude', 'longitude', 'number',
        'tp_mm', 'subcuenca_id', 'subcuenca_nombre', 'source_table', 'processed_at', 'updated_at',
    )
)

DeltaTable.forName(spark, TARGET_TABLE).alias('t').merge(
    silver_df.alias('s'),
    't.run_date = s.run_date AND t.run_time = s.run_time AND t.step_hours = s.step_hours AND t.latitude = s.latitude AND t.longitude = s.longitude AND t.number <=> s.number',
).whenNotMatchedInsertAll().execute()

spark.table(TARGET_TABLE).agg(F.min('run_date').alias('inicio'), F.max('run_date').alias('fin'), F.count('*').alias('rows')).show()